1 script - Utilizacao do modelo. Deve ser a primeira celula a ser executada apos ter os pesos treinados do modelo que estiver sendo utilizado. O script retorna uma planilha com as predicoes realizadas pelo modelo.

In [3]:
from ultralytics import YOLO
import cv2
import numpy as np

import util
from sort.sort import Sort
from util import get_car, read_license_plate, write_csv



results = {}

mot_tracker = Sort()

# load models
coco_model = YOLO('yolov8n.pt')
license_plate_detector = YOLO('/home/messyas/ml/jetson/placas-model/models/runs/yolov8n_lp/weights/best.pt')

# load video
cap = cv2.VideoCapture('./sample.mp4')

vehicles = [2, 3, 5, 7]

# read frames
frame_nmr = -1
ret = True
while ret:
    frame_nmr += 1
    ret, frame = cap.read()
    if ret:
        results[frame_nmr] = {}
        # detect vehicles
        detections = coco_model(frame)[0]
        detections_ = []
        for detection in detections.boxes.data.tolist():
            x1, y1, x2, y2, score, class_id = detection
            if int(class_id) in vehicles:
                detections_.append([x1, y1, x2, y2, score])

        # track vehicles
        track_ids = mot_tracker.update(np.asarray(detections_))

        # detect license plates
        license_plates = license_plate_detector(frame)[0]
        for license_plate in license_plates.boxes.data.tolist():
            x1, y1, x2, y2, score, class_id = license_plate

            # assign license plate to car
            xcar1, ycar1, xcar2, ycar2, car_id = get_car(license_plate, track_ids)

            if car_id != -1:

                # crop license plate
                license_plate_crop = frame[int(y1):int(y2), int(x1): int(x2), :]

                # process license plate
                license_plate_crop_gray = cv2.cvtColor(license_plate_crop, cv2.COLOR_BGR2GRAY)
                _, license_plate_crop_thresh = cv2.threshold(license_plate_crop_gray, 64, 255, cv2.THRESH_BINARY_INV)

                # read license plate number
                license_plate_text, license_plate_text_score = read_license_plate(license_plate_crop_thresh)

                if license_plate_text is not None:
                    results[frame_nmr][car_id] = {'car': {'bbox': [xcar1, ycar1, xcar2, ycar2]},
                                                  'license_plate': {'bbox': [x1, y1, x2, y2],
                                                                    'text': license_plate_text,
                                                                    'bbox_score': score,
                                                                    'text_score': license_plate_text_score}}

# write results
write_csv(results, './test.csv')


0: 384x640 21 cars, 1 bus, 2 trucks, 4.4ms
Speed: 2.0ms preprocess, 4.4ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 License_Plates, 6.1ms
Speed: 1.5ms preprocess, 6.1ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 23 cars, 1 bus, 2 trucks, 4.4ms
Speed: 1.4ms preprocess, 4.4ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 License_Plates, 6.0ms
Speed: 1.8ms preprocess, 6.0ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 22 cars, 1 bus, 2 trucks, 5.1ms
Speed: 1.4ms preprocess, 5.1ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 License_Plates, 4.7ms
Speed: 1.2ms preprocess, 4.7ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 23 cars, 1 bus, 2 trucks, 4.5ms
Speed: 1.3ms preprocess, 4.5ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 License_Plates, 4.3ms
Speed: 1